<!-- nav -->
[← 6 · Calls, members and subscripts](06_Calls_Members_And_Subscripts.ipynb) · [Home](../../README.md) · [8 · Sublanguages →](08_00_Sublanguages.ipynb)

# 7 · Bindings

Tutorial 4 showed a name as a reference: a term that reads `x` finds its value in its scope. Some expressions give a
name its value themselves: `let c = ... in ...`, a lambda's parameters, a quantifier's item, a comprehension's `for`.
Each **binds** a name within part of itself, so the names it binds are not names its scope must give.

Every language writes these differently, and each dialect models them with kinds of its own, but every kind that binds
names is a variation of one, the framework's `let`: it refines `Let`, whose `name`, `value` and `body` it holds first,
so that what it binds, from what and where reads alike in every language
([CAP-279: names an expression binds found alike in every language](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-279.md)).
This notebook builds a `let`, then lambdas, quantifiers and comprehensions, each in the builders' calls beside its
language's text, and asks each what it needs from its scope with `Symbolics.free` (tutorial 4).

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `variable` writes a
name in the dialects whose names are the framework's `variable`, and `translate` translates an expression from one
dialect to another, as tutorial 6 does; `text` prints a term, or `-` for an empty subterm. Where a model is Python's, Python also reads its text, with
`Python3.Text.FromText`, and checks that it gives the same term (tutorial 3).

In [1]:
from mbse.Expressions import Translators
from mbse.Expressions.Patterns.Expressions import Expressions as E
from mbse.Expressions.Python import Python3
from mbse.Expressions.TypeScript import TypeScript5
from mbse.Expressions.C import Expressions as C
from mbse.Expressions.SystemVerilog import Expressions as SystemVerilog
from mbse.Expressions.Excel import Expressions as Excel
from mbse.Expressions.MATLAB import Expressions as Matlab
from mbse.Expressions.LaTeX import Expressions as Latex
from mbse.Expressions.Framework import Symbolics, Terms
from toolkit import dialects, show

store = dialects()
S, Term = store.Expressions, store.Expressions.Framework.Term
amd64 = store.Instances.Expressions.SystemV.AMD64
variable = lambda name: lambda d: d.variable(lambda v: v.name(name))


def translate(expression, source, target):
    """`expression`, of the dialect `source`, translated to `target`, written as a term."""
    with store.transaction():
        return Term(Translators.between(source, target).forward(expression))


def text(printer, term):
    """A term's text as `printer` writes it, or `-` for an empty subterm."""
    return "-" if term is None else printer(term)

<a id="T07.1"></a>
## 07.1 · let

Patterns' `let c = get(s, 'celsius') in and(ge(c, 2.0), le(c, 8.0))` reads a shipment's temperature once, names it
`c`, and checks it twice. A `let` is a term of the framework's kind `let`, the schema `Expressions.Framework.Let`: its
`name`, the name it binds, an attribute; its `value`, the term whose value the name takes; and its `body`, the term
the name is bound in. Its value and body are the framework's terms, each held in its dialect's branch (`t.patterns`):

In [2]:
with store.transaction():
    let = S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.let(lambda l: l.name("c")
        .value(lambda t: t.patterns(lambda d: d.call(lambda c: c.callable("get").arguments([
            lambda t: t.patterns(variable("s")),
            lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar("celsius"))))]))))
        .body(lambda t: t.patterns(lambda d: d.call(lambda c: c.callable("and").arguments([
            lambda t: t.patterns(lambda d: d.call(lambda m: m.callable("ge").arguments([
                lambda t: t.patterns(variable("c")),
                lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(2.0))))]))),
            lambda t: t.patterns(lambda d: d.call(lambda m: m.callable("le").arguments([
                lambda t: t.patterns(variable("c")),
                lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(8.0))))])))])))))))
form = Terms.form(Terms.term(let))
value, body = form.arguments
print(show(let))
print(form.kind, "name:", form.attributes["name"], " value:", show(value), " body:", show(body))

let c = get(s, 'celsius') in and(ge(c, 2.0), le(c, 8.0))
let name: c  value: get(s, 'celsius')  body: and(ge(c, 2.0), le(c, 8.0))


`c` is bound in the body alone. The body needs `c`, which the `let` gives it; the value needs `s`, which nothing in the
term gives. So the whole term needs `s` from its scope, and `validate` reports `s`, never `c`:

In [3]:
for part, label in ((value, "value"), (body, "body"), (Terms.term(let), "let")):
    print(f"{label:5} needs", ", ".join(sorted(Symbolics.free(part, E.DIALECT))))
print(E.DIALECT.validate(let))
print(E.DIALECT.validate(let, bound=["s"]))
assert Symbolics.free(let, E.DIALECT) == {"s"}

value needs s
body  needs c
let   needs s
["value: argument 0: variable 's' is not bound"]
[]


<a id="T07.2"></a>
## 07.2 · let in each language

Three languages have a `let` of their own, and each dialect holds it as the framework's `Let` itself, its value and body
that dialect's terms:

- **Python3** has no `let` expression, but a lambda applied at once is one, `(lambda c: ...)(s.celsius)`: Python3's
  `let` is printed so, and Python reads that text back as a `let`.
- **Excel**'s `LET(c, s.celsius, ...)`, since Excel 365.
- **LaTeX**'s `... \quad \text{where } c = ...`, the way a paper writes it.

In [4]:
with store.transaction():
    lets = {
        "Python3": (Python3, S.Python.Python3.Expression(lambda e: e.root(lambda u: u.let(lambda l: l.name("c")
            .value(lambda t: t.python3(lambda d: d.attribute(lambda a: a.value(variable("s")).attr("celsius"))))
            .body(lambda t: t.python3(lambda d: d.boolop(lambda b: b.op("and").values([
                lambda v: v.compare(lambda c: c.left(variable("c")).comparisons([lambda k: k.op(">=").comparator(
                    lambda t: t.literal(lambda r: r.value(lambda v: v.as_scalar(2.0)).text("2.0")))])),
                lambda v: v.compare(lambda c: c.left(variable("c")).comparisons([lambda k: k.op("<=").comparator(
                    lambda t: t.literal(lambda r: r.value(lambda v: v.as_scalar(8.0)).text("8.0")))]))])))))))),
        "Excel": (Excel, S.Excel.Expressions.Expression(lambda e: e.root(lambda u: u.let(lambda l: l.name("c")
            .value(lambda t: t.excel(lambda d: d.field(lambda f: f.value(variable("s")).name("celsius"))))
            .body(lambda t: t.excel(lambda d: d.call(lambda c: c.callable("AND").arguments([
                lambda t: t.excel(lambda d: d.infix(lambda i: i.left(variable("c")).operator(">=")
                    .right(lambda r: r.literal(lambda r: r.value(lambda v: v.as_scalar(2.0)))))),
                lambda t: t.excel(lambda d: d.infix(lambda i: i.left(variable("c")).operator("<=")
                    .right(lambda r: r.literal(lambda r: r.value(lambda v: v.as_scalar(8.0))))))])))))))),
        "LaTeX": (Latex, S.LaTeX.Expressions.Expression(lambda e: e.root(lambda u: u.let(lambda l: l.name("c")
            .value(lambda t: t.latex(lambda d: d.member(lambda m: m.value(variable("s")).name("celsius"))))
            .body(lambda t: t.latex(lambda d: d.binary(lambda b: b.operator("\\land")
                .left(lambda o: o.binary(lambda b: b.left(variable("c")).operator("\\geq")
                    .right(lambda r: r.literal(lambda r: r.value(lambda v: v.as_scalar(2.0))))))
                .right(lambda o: o.binary(lambda b: b.left(variable("c")).operator("\\leq")
                    .right(lambda r: r.literal(lambda r: r.value(lambda v: v.as_scalar(8.0))))))))))))),
    }
    read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("(lambda c: c >= 2.0 and c <= 8.0)(s.celsius)")))
for language, (dialect, built) in lets.items():
    written = dialect.Text.ToText(built)
    assert written == dialect.Text.ToText(translate(let, E.DIALECT, dialect.Expressions.DIALECT))  # as the translator writes it
    needs = ", ".join(sorted(Symbolics.free(built, dialect.Expressions.DIALECT)))
    print(f"{language:8} {Terms.form(Terms.term(built)).kind} needs {needs}: {written}")
assert [dialect.Text.ToText(built) for dialect, built in lets.values()] == [
    "(lambda c: c >= 2.0 and c <= 8.0)(s.celsius)", "=LET(c, s.celsius, AND(c >= 2.0, c <= 8.0))",
    "c \\geq 2.0 \\land c \\leq 8.0 \\quad \\text{where } c = s.\\mathit{celsius}"]
assert Terms.same(read, lets["Python3"][1])  # Python reads its own text into the same let

Python3  let needs s: (lambda c: c >= 2.0 and c <= 8.0)(s.celsius)
Excel    let needs s: =LET(c, s.celsius, AND(c >= 2.0, c <= 8.0))
LaTeX    let needs s: c \geq 2.0 \land c \leq 8.0 \quad \text{where } c = s.\mathit{celsius}


C, SystemVerilog and MATLAB have no expression that binds a name to a value. The translator writes the value where the
body reads the name, so the text means the same, but it is no longer a `let`: the value is read twice, and the tree that
comes back to Patterns is the substituted one, not the `let`:

In [5]:
for dialect in (C, SystemVerilog, Matlab):
    target = dialect.Expressions.DIALECT
    there = translate(let, E.DIALECT, target)
    back = translate(there, target, E.DIALECT)
    print(f"{target.name():13} {Terms.form(there).kind:16} {dialect.Text.ToText(there)}")
    print(f"{'':13} back: {show(back)}")
    assert not Terms.same(back, let)

C             binaryexpression s.celsius >= 2.0 && s.celsius <= 8.0
              back: and(ge(get(s, 'celsius'), 2.0), le(get(s, 'celsius'), 8.0))
SystemVerilog binaryexpression s.celsius >= 2.0 && s.celsius <= 8.0
              back: and(ge(get(s, 'celsius'), 2.0), le(get(s, 'celsius'), 8.0))
MATLAB        binary           s.celsius >= 2.0 && s.celsius <= 8.0
              back: and(ge(get(s, 'celsius'), 2.0), le(get(s, 'celsius'), 8.0))


<a id="T07.3"></a>
## 07.3 · A lambda

A lambda binds its parameters in its body: Python's `lambda x: x + k` and TypeScript's `(x) => x + k` each need only
`k` from their scope. Python3's `lambda` and TypeScript5's `arrowfunctionexpression` each refine `Let`: the body is
Let's `body`, and the parameters, which a `let`'s one `name` can't hold, are their own (Python3's `args`, an `arguments`
helper; TypeScript5's `params`, its patterns). A lambda's value is given when it is called, not when it is written, so
Let's `name` and `value` are left unset, and its form gives None as its value subterm (`-` below):

In [6]:
with store.transaction():
    lambdas = {
        "Python3": (Python3, S.Python.Python3.Expression(lambda e: e.root(lambda u: getattr(u, "lambda")(lambda l: l
            .args(lambda a: a.args([lambda g: g.arg("x")]))
            .body(lambda b: b.binop(lambda o: o.left(variable("x")).op("+").right(variable("k")))))))),
        "TypeScript5": (TypeScript5, S.TypeScript.TypeScript5.Expression(lambda e: e.root(
            lambda u: u.arrowfunctionexpression(lambda l: l
                .params([lambda p: p.identifier(lambda i: i.name("x"))])
                .body(lambda b: b.binaryexpression(lambda o: o
                    .left(lambda t: t.identifier(lambda i: i.name("x"))).operator("+")
                    .right(lambda t: t.identifier(lambda i: i.name("k"))))))))),
    }
    read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("lambda x: x + k")))
for language, (dialect, built) in lambdas.items():
    form = Terms.form(Terms.term(built))
    print(f"{language:12} {dialect.Text.ToText(built):16} {form.kind:23} name: {form.attributes.get('name') or '-'}, value:",
          text(dialect.Text.ToText, form.arguments[0]), " needs", ", ".join(sorted(Symbolics.free(built, dialect.Expressions.DIALECT))))
assert [dialect.Text.ToText(built) for dialect, built in lambdas.values()] == ["lambda x: x + k", "(x) => x + k"]
assert Terms.same(read, lambdas["Python3"][1])  # Python reads its own text into the same lambda

Python3      lambda x: x + k  lambda                  name: -, value: -  needs k
TypeScript5  (x) => x + k     arrowfunctionexpression name: -, value: -  needs k


Validation, which reads the same scopes, reports `k` and not `x`, in each language's words:

In [7]:
for language, (dialect, built) in lambdas.items():
    print(f"{language:12}", dialect.Expressions.DIALECT.validate(built), dialect.Expressions.DIALECT.validate(built, bound=["k"]))

Python3      ["body: right: variable 'k' is not bound"] []
TypeScript5  ["body: right: identifier 'k' is not bound"] []


<a id="T07.4"></a>
## 07.4 · Quantifiers bind too

Patterns' `all(get(s, 'readings'), c => le(c, 8.0))` asks whether every reading is at most 8.0: a `quantifier` binds its
`name`, `c`, to each item of a collection in turn, folds its `body` for each, and combines the results by its
`quantifier`, `all` (case study 11). It refines `Let`: the collection is Let's `value`, the term the name takes its
values from. Unlike `let`'s, its value and body are Patterns' own terms, not the framework's, so they need no `t.patterns`
branch:

In [8]:
with store.transaction():
    every = S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.quantifier(lambda q: q
        .quantifier("all").name("c")
        .value(lambda d: d.call(lambda c: c.callable("get").arguments([
            lambda t: t.patterns(variable("s")),
            lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar("readings"))))])))
        .body(lambda d: d.call(lambda c: c.callable("le").arguments([
            lambda t: t.patterns(variable("c")),
            lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(8.0))))]))))))
form = Terms.form(Terms.term(every))
print(show(every))
print(form.kind, "name:", form.attributes["name"], " quantifier:", form.attributes["quantifier"], " value:",
      show(form.arguments[0]), " body:", show(form.arguments[1]))
print("needs", ", ".join(sorted(Symbolics.free(every, E.DIALECT))))

all(get(s, 'readings'), c => le(c, 8.0))
quantifier name: c  quantifier: all  value: get(s, 'readings')  body: le(c, 8.0)
needs s


Each language that can say "every reading" says it with a binder of its own, and each refines `Let` too, its collection
Let's `value`:

- **Python**: `all(c <= 8.0 for c in s.readings)`, a call of `all` on a generator expression (07.5), whose `for`
  clause binds `c`.
- **Excel**: `AND(MAP(s.readings, LAMBDA(c, c <= 8.0)))`: `MAP` and its `LAMBDA` are one kind, `map`, which binds
  the lambda's parameter to each item.
- **MATLAB**: `all(arrayfun(@(c) c <= 8.0, s.readings))`: `arrayfun` and its anonymous function are one kind,
  `arrayfun`.
- **SystemVerilog**: `s.readings.and(c) with (c <= 8.0)`, an array reduction with an iterator,
  `arraymethodwithexpression`: its value is the method's call, `s.readings.and(c)`, which names the iterator `c`, so
  it leaves Let's `name` unset, and its scopes say what it binds.

Each is the translation of Patterns' quantifier, and translates back to it:

In [9]:
with store.transaction():
    quantified = {
        "Python3": (Python3, S.Python.Python3.Expression(lambda e: e.root(lambda u: u.call(lambda c: c
            .func(variable("all"))
            .args([lambda a: a.generatorexp(lambda g: g
                .body(lambda b: b.compare(lambda c: c.left(variable("c")).comparisons([lambda k: k.op("<=").comparator(
                    lambda t: t.literal(lambda r: r.value(lambda v: v.as_scalar(8.0)).text("8.0")))])))
                .generators([lambda f: f.target(variable("c"))
                    .iter(lambda i: i.attribute(lambda a: a.value(variable("s")).attr("readings"))).ifs([])]))]))))),
        "Excel": (Excel, S.Excel.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c
            .callable("AND").arguments([lambda t: t.excel(lambda d: d.map(lambda m: m.name("c")
                .value(lambda v: v.field(lambda f: f.value(variable("s")).name("readings")))
                .body(lambda b: b.infix(lambda i: i.left(variable("c")).operator("<=")
                    .right(lambda r: r.literal(lambda r: r.value(lambda v: v.as_scalar(8.0))))))))]))))),
        "MATLAB": (Matlab, S.MATLAB.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c
            .callable("all").arguments([lambda t: t.matlab(lambda d: d.arrayfun(lambda m: m.name("c")
                .value(lambda v: v.field(lambda f: f.value(variable("s")).name("readings")))
                .body(lambda b: b.binary(lambda i: i.left(variable("c")).operator("<=")
                    .right(lambda r: r.literal(lambda r: r.value(lambda v: v.as_scalar(8.0))))))))]))))),
        "SystemVerilog": (SystemVerilog, S.SystemVerilog.Expressions.Expression(lambda e: e.root(
            lambda u: u.arraymethodwithexpression(lambda m: m
                .value(lambda v: v.callexpression(lambda c: c
                    .callee(lambda f: f.memberexpression(lambda m: m
                        .value(lambda v: v.memberexpression(lambda r: r
                            .value(lambda n: n.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("s")))))
                            .member(lambda i: i.spelling("readings"))))
                        .member(lambda i: i.spelling("and"))))
                    .arguments([lambda a: a.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("c"))))])))
                .body(lambda b: b.binaryexpression(lambda o: o
                    .left(lambda l: l.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("c")))))
                    .operator("<=").right(lambda r: r.literal(lambda r: r.value(lambda v: v.as_scalar(8.0)))))))))),
    }
    read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("all(c <= 8.0 for c in s.readings)")))
for language, (dialect, built) in quantified.items():
    target = dialect.Expressions.DIALECT
    there = translate(every, E.DIALECT, target)
    assert dialect.Text.ToText(there) == dialect.Text.ToText(built)  # as the translator writes it
    assert Terms.same(translate(built, target, E.DIALECT), every)  # and back
    binder = next(t for t in Terms.walk(Terms.term(built)) if Terms.is_binder(Terms.kind_of(t)[1].as_schema()))
    print(f"{language:13} {dialect.Text.ToText(built):42} {Terms.form(binder).kind:25} needs",
          ", ".join(sorted(Symbolics.free(built, target))))
assert [dialect.Text.ToText(built) for dialect, built in quantified.values()] == [
    "all(c <= 8.0 for c in s.readings)", "=AND(MAP(s.readings, LAMBDA(c, c <= 8.0)))",
    "all(arrayfun(@(c) c <= 8.0, s.readings))", "s.readings.and(c) with (c <= 8.0)"]
assert Terms.same(read, quantified["Python3"][1])  # Python reads its own text into the same call

Python3       all(c <= 8.0 for c in s.readings)          generatorexp              needs s
Excel         =AND(MAP(s.readings, LAMBDA(c, c <= 8.0))) map                       needs s
MATLAB        all(arrayfun(@(c) c <= 8.0, s.readings))   arrayfun                  needs s


SystemVerilog s.readings.and(c) with (c <= 8.0)          arraymethodwithexpression needs s


C and LaTeX have no counterpart for a quantifier yet, a known gap
([CAP-102: collections translated to LaTeX and C and C++](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-102.md)):
C's dialect has no expression that visits a collection, and LaTeX's no kind for `\forall`. The translators say so,
rather than guess:

In [10]:
for dialect in (C, Latex):
    try:
        translate(every, E.DIALECT, dialect.Expressions.DIALECT)
    except ValueError as error:
        print(f"{dialect.Expressions.DIALECT.name():6}", error)

C      Patterns quantifier 'all' has no C counterpart
LaTeX  Patterns quantifier 'all' has no LaTeX counterpart


<a id="T07.5"></a>
## 07.5 · Comprehensions and named expressions

Python's `[p.pin for p in ports if p.pin > 0]` is a `listcomp`: its element, `p.pin`, is Let's `body`, and its `for`
clauses are its `generators`, each a `comprehension` helper holding a `target` (a pattern, which tutorial 8 shows),
the `iter` it takes values from and the conditions, `ifs`, that filter them. The target is bound in the conditions, in
later clauses and in the element, not in its own iterable, so the comprehension needs `ports` alone. A comprehension may
have several clauses, so Let's one `name` and `value` are left unset. Python's `set`, `dict` and generator comprehensions
are alike:

In [11]:
with store.transaction():
    pins = S.Python.Python3.Expression(lambda e: e.root(lambda u: u.listcomp(lambda l: l
        .body(lambda b: b.attribute(lambda a: a.value(variable("p")).attr("pin")))
        .generators([lambda g: g.target(variable("p")).iter(variable("ports")).ifs([
            lambda i: i.compare(lambda c: c.left(lambda t: t.attribute(lambda a: a.value(variable("p")).attr("pin")))
                .comparisons([lambda k: k.op(">").comparator(
                    lambda t: t.literal(lambda r: r.value(lambda v: v.as_scalar(0)).text("0").base(10)))]))])]))))
    read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("[p.pin for p in ports if p.pin > 0]")))
form = Terms.form(Terms.term(pins))
print(Python3.Text.ToText(pins), "is a", form.kind)
print("name:", form.attributes.get("name") or "-", " value:", text(Python3.Text.ToText, form.arguments[0]), " body:",
      text(Python3.Text.ToText, form.arguments[1]))
print("needs", ", ".join(sorted(Symbolics.free(pins, Python3.Expressions.DIALECT))))
assert Python3.Text.ToText(pins) == "[p.pin for p in ports if p.pin > 0]" and Terms.same(read, pins)
assert Symbolics.free(pins, Python3.Expressions.DIALECT) == {"ports"}

[p.pin for p in ports if p.pin > 0] is a listcomp
name: -  value: -  body: p.pin
needs ports


A **named expression**, `(y := f(x))`, binds `y` to its value, `f(x)`, and gives that value. Its `target` is a name
of the enclosing scope, not of the expression's own: `y` is bound for the code after it, which the term does not hold.
So `namedexpr` holds its value as Let's `value`, leaves Let's `body` unset, and needs only what its value needs:

In [12]:
with store.transaction():
    walrus = S.Python.Python3.Expression(lambda e: e.root(lambda u: u.namedexpr(lambda n: n
        .target(variable("y"))
        .value(lambda v: v.call(lambda c: c.func(variable("f")).args([variable("x")]))))))
    read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("(y := f(x))")))
form = Terms.form(Terms.term(walrus))
print(Python3.Text.ToText(walrus), "is a", form.kind)
print("value:", text(Python3.Text.ToText, form.arguments[0]), " body:", text(Python3.Text.ToText, form.arguments[1]))
print("needs", ", ".join(sorted(Symbolics.free(walrus, Python3.Expressions.DIALECT))))
assert Python3.Text.ToText(walrus) == "(y := f(x))" and Terms.same(read, walrus)

(y := f(x)) is a namedexpr
value: f(x)  body: -
needs f, x


<a id="T07.6"></a>
## 07.6 · Every binder a variation of let

Every kind that binds names, in every dialect, refines the framework's `Let`, and lists Let's `name`, `value` and `body`
first, where a language's own syntax tree called them something else: Patterns' quantifier held its collection as
`collection`, MATLAB's `arrayfun` and Excel's `map` theirs as `array`, and Python's comprehensions their element as
`elt`; each is now Let's `value` or `body`. A kind leaves unset what it has no name or term for. Each still binds as its
kind says: a quantifier its name to each item, an import what it imports, and a kind with scopes of its own (a lambda's
parameters, a comprehension's targets, an iterator) by them.

Which kinds those are is data, read from the store: each dialect's branch of the framework's `Term` holds its kinds, and
each binder's schema refines `Expressions.Framework.Let`, which a dialect's `let` is itself. A dialect that reads
another's kinds (Python Types reads Python3's) lists them too, so each schema is shown once, under the dialect that
defines it:

In [13]:
LET = store.registered("Expressions.Framework.Let")
binders = {}  # each binder's schema, once, by the dialect that defines it
for held in store.registered("Expressions.Framework.Term").branches:  # each dialect's terms
    for kind in held.schema.branches:
        if Terms.is_binder(kind.schema):
            assert kind.schema is LET or kind.schema.refines is LET
            assert list(kind.schema.properties)[:3] == ["name", "value", "body"]
            owner = "Framework" if kind.schema is LET else kind.schema.name.split(".", 1)[1].rsplit(".", 1)[0]
            binders.setdefault(owner, {})[kind.name] = kind.schema
for owner, kinds in binders.items():
    print(f"{owner:28}", " ".join(kinds))
assert list(binders) == ["Framework", "Patterns.Expressions", "Python.Python3", "TypeScript.TypeScript5", "MATLAB.Expressions",
                         "Excel.Expressions", "Cpp.Expressions", "SystemVerilog.Expressions", "SystemVerilog.Constraints",
                         "TypeScript.Types"]

Framework                    let
Patterns.Expressions         quantifier
Python.Python3               listcomp setcomp dictcomp generatorexp namedexpr lambda import importfrom
TypeScript.TypeScript5       functionexpression arrowfunctionexpression classexpression tsemptybodyfunctionexpression
MATLAB.Expressions           arrayfun import
Excel.Expressions            map
Cpp.Expressions              lambdaexpression requiresexpression
SystemVerilog.Expressions    arraymethodwithexpression
SystemVerilog.Constraints    foreachconstraint
TypeScript.Types             tsfunctiontype tsconstructortype tsmappedtype


C has none: none of C's expressions binds a name. C++'s `lambdaexpression` and `requiresexpression` bind in a body of
statements that is not part of the term, so they leave all three of Let's properties unset. TypeScript's function and
mapped types, SystemVerilog's `foreach` constraint and the imports are binders too, each in its own dialect.

So the framework reads every binder alike. The binders this notebook built, in six languages, each give their form
Let's `name`, and Let's `value` and `body` first, `-` here where a kind leaves them unset, and `Symbolics.free` reads what
each binds through its kind:

In [14]:
built = [("Patterns", show, let), ("Patterns", show, every), ("Python3", Python3.Text.ToText, lets["Python3"][1]),
         ("Excel", Excel.Text.ToText, lets["Excel"][1]), ("LaTeX", Latex.Text.ToText, lets["LaTeX"][1]),
         *((language, dialect.Text.ToText, built) for language, (dialect, built) in lambdas.items()),
         *((language, dialect.Text.ToText, built) for language, (dialect, built) in quantified.items()),
         ("Python3", Python3.Text.ToText, pins), ("Python3", Python3.Text.ToText, walrus)]
print(f"{'':13} {'kind':25} {'name':5} {'value':22} body")
for language, printer, expression in built:
    for binder in (t for t in Terms.walk(Terms.term(expression)) if Terms.is_binder(Terms.kind_of(t)[1].as_schema())):
        form = Terms.form(binder)
        value, body = form.arguments[:2]
        print(f"{language:13} {form.kind:25} {form.attributes.get('name') or '-':5} {text(printer, value):22}",
              text(printer, body))

              kind                      name  value                  body
Patterns      let                       c     get(s, 'celsius')      and(ge(c, 2.0), le(c, 8.0))
Patterns      quantifier                c     get(s, 'readings')     le(c, 8.0)
Python3       let                       c     s.celsius              c >= 2.0 and c <= 8.0
Excel         let                       c     =s.celsius             =AND(c >= 2.0, c <= 8.0)
LaTeX         let                       c     s.\mathit{celsius}     c \geq 2.0 \land c \leq 8.0
Python3       lambda                    -     -                      x + k
TypeScript5   arrowfunctionexpression   -     -                      x + k
Python3       generatorexp              -     -                      c <= 8.0
Excel         map                       c     =s.readings            =c <= 8.0
MATLAB        arrayfun                  c     s.readings             c <= 8.0
SystemVerilog arraymethodwithexpression -     s.readings.and(c)      c <= 8.0
Pyth

## Why it works this way

- **One kind of binding, many syntaxes.** A `let`, a lambda, a quantifier, a comprehension and an array method's
  iterator each give a name a value within part of themselves. Modeled as variations of one kind, each reads alike to
  every tool: what it binds, from what, and where.
- **Each language keeps its own kind.** A dialect still models its language's syntax as that language has it (Python's
  comprehension clauses, SystemVerilog's iterator in its call), so its text prints back as written; refining `Let` adds
  no syntax, only says which of its parts are Let's.
- **Free names say what data a constraint needs.** What a term binds is not what its scope must give, so
  `Symbolics.free` and validation ask the scope only for the rest, in every language alike.
- **A translator that can't keep a binding says so, or keeps its meaning.** Where a language has no `let`, the value is
  written in place of the name; where it has no quantifier, the translator refuses rather than guess.

<!-- nav -->
[← 6 · Calls, members and subscripts](06_Calls_Members_And_Subscripts.ipynb) · [Home](../../README.md) · [8 · Sublanguages →](08_00_Sublanguages.ipynb)